# Notebook 03 - Recommandation Semantique

**MixCraft** - EFREI M1 DE&IA - Adam Beloucif & Emilien Morice - Janvier 2026

## Objectifs

1. Implementer le moteur de recommandation par cosinus
2. Evaluer les metriques Precision@K, Recall@K, NDCG@K
3. Comparer TF-IDF baseline vs SBERT
4. Valider sur un jeu de requetes test

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import load_all_datasets
from src.embeddings import EmbeddingEngine
from src.recommender import CocktailRecommender, precision_at_k, recall_at_k, ndcg_at_k

EFREI_NAVY = '#163767'
EFREI_PINK = '#FF43B8'
EFREI_BLUE = '#0C78B4'

df = load_all_datasets()
print(f'Corpus : {len(df)} cocktails')

## 1. Indexation du corpus

In [ ]:
engine = EmbeddingEngine(cache=True)
rec = CocktailRecommender(engine=engine)
rec.fit(df)
print('Moteur indexe.')
print(f'Embeddings shape : {rec._embeddings.shape}')

## 2. Tests de recommandation

In [ ]:
test_queries = [
    'something fresh and fruity with citrus',
    'strong and bitter classic cocktail',
    'sweet tropical summer drink',
    'gin based refreshing cocktail',
    'creamy and smooth dessert cocktail',
]

for query in test_queries:
    print(f'\nRequete : "{query}"')
    results = rec.recommend_by_query(query, top_k=3)
    for r in results:
        print(f'  {r.rank}. {r.name:<30} (score: {r.similarity_score:.3f}) [{r.category}]')

## 3. Recommandation par ingredients

In [ ]:
available_ingredients = ['vodka', 'lime juice', 'ginger beer', 'mint']
print(f'Ingredients disponibles : {available_ingredients}\n')
results_ing = rec.recommend_by_ingredients(available_ingredients, top_k=5)
for r in results_ing:
    print(f'{r.rank}. {r.name:<30} score={r.similarity_score:.3f}')
    print(f'   Ingredients: {r.ingredients[:80]}...')

## 4. Comparaison SBERT vs TF-IDF baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF baseline
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(df['text_full'].fillna(''))

# Jeu de test synthetique : 20 requetes avec cocktails pertinents connus
np.random.seed(42)
test_set = []
for i in range(20):
    idx = np.random.randint(0, len(df))
    row = df.iloc[idx]
    test_set.append({
        'query': f'{row["category"]} with {row["ingredients"].split(",")[0]}',
        'relevant_names': {row['name']}
    })

# Evaluation SBERT
results_sbert = {'P@5': [], 'R@5': [], 'NDCG@5': []}
results_tfidf = {'P@5': [], 'R@5': [], 'NDCG@5': []}

for item in test_set:
    query = item['query']
    relevant = item['relevant_names']

    # SBERT
    sbert_recs = rec.recommend_by_query(query, top_k=5)
    sbert_names = [r.name for r in sbert_recs]
    results_sbert['P@5'].append(precision_at_k(relevant, sbert_names, 5))
    results_sbert['R@5'].append(recall_at_k(relevant, sbert_names, 5))
    results_sbert['NDCG@5'].append(ndcg_at_k(relevant, sbert_names, 5))

    # TF-IDF
    q_vec = vectorizer.transform([query])
    cos_scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top5_idx = cos_scores.argsort()[::-1][:5]
    tfidf_names = df.iloc[top5_idx]['name'].tolist()
    results_tfidf['P@5'].append(precision_at_k(relevant, tfidf_names, 5))
    results_tfidf['R@5'].append(recall_at_k(relevant, tfidf_names, 5))
    results_tfidf['NDCG@5'].append(ndcg_at_k(relevant, tfidf_names, 5))

# Resume
print('=== Comparaison TF-IDF vs SBERT ===')
metrics = ['P@5', 'R@5', 'NDCG@5']
for m in metrics:
    s = np.mean(results_sbert[m])
    t = np.mean(results_tfidf[m])
    gain = (s - t) / max(t, 1e-9) * 100
    print(f'{m:10s}  TF-IDF={t:.3f}  SBERT={s:.3f}  gain={gain:+.1f}%')

In [ ]:
# Graphique comparatif
metrics_list = ['P@5', 'R@5', 'NDCG@5']
sbert_means = [np.mean(results_sbert[m]) for m in metrics_list]
tfidf_means = [np.mean(results_tfidf[m]) for m in metrics_list]

x = np.arange(len(metrics_list))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, tfidf_means, width, label='TF-IDF baseline', color='#a8bdd4', edgecolor='white')
bars2 = ax.bar(x + width/2, sbert_means, width, label='SBERT all-MiniLM-L6-v2', color=EFREI_NAVY, edgecolor='white')

for bar in bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                textcoords='offset points', ha='center', fontsize=10, fontweight='bold', color=EFREI_NAVY)

ax.set_title('Comparaison TF-IDF vs SBERT (jeu test 20 requetes)', fontsize=14, fontweight='bold', color=EFREI_NAVY)
ax.set_xticks(x)
ax.set_xticklabels(metrics_list, fontsize=12)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../assets/fig_sbert_vs_tfidf.png', dpi=150, bbox_inches='tight')
plt.show()